In [ ]:
import polars as pl
from definitions import ROOT_DIR
from datetime import date
# import hftbacktest as hft
import numpy as np

def get_data_path(d: date, kind: str) -> str:
    return str(ROOT_DIR / f"data/{kind}/binance_futures/BTCUSDT/{d.strftime('%Y-%m-%d')}.parquet")

df_ob = pl.read_parquet(get_data_path(date(2025, 7, 1), "orderbook"))
df_tr = pl.read_parquet(get_data_path(date(2025, 7, 1), "trades"))

In [ ]:
df_ob = df_ob.with_columns((df_ob["event_time"] * int(1e6)).alias("event_time"))
df_tr = df_tr.with_columns((df_tr["event_time"] * int(1e6)).alias("event_time"))

In [ ]:
df_ob.head(1000)

In [ ]:
from hftbacktest import BUY_EVENT, SELL_EVENT, DEPTH_EVENT, DEPTH_SNAPSHOT_EVENT, EXCH_EVENT, LOCAL_EVENT, event_dtype
from hftbacktest.data import correct_event_order, validate_event_order

def get_hftbacktest_array(df: pl.DataFrame) -> np.ndarray:
    # Create empty array of the right size
    arr = np.zeros(len(df), dtype=event_dtype)
    
    # Use vectorized operations to determine event types
    is_snapshot = (df["event_type"] == "snapshot").to_numpy()
    is_ask = (df["side"] == "ask").to_numpy()
    
    # Set event type values with numpy's where (no loops)
    arr["ev"] = np.where(
        is_snapshot,
        DEPTH_SNAPSHOT_EVENT,
        DEPTH_EVENT | np.where(is_ask, SELL_EVENT, BUY_EVENT) | EXCH_EVENT | LOCAL_EVENT
    )
    
    # Directly set other fields from the dataframe
    arr["exch_ts"] = df["event_time"].to_numpy()
    arr["local_ts"] = df["received_time"].to_numpy()
    arr["px"] = df["price"].cast(pl.Float64).to_numpy()
    arr["qty"] = df["quantity"].cast(pl.Float64).to_numpy()

    print("Correcting event order")
    arr = correct_event_order(
        arr, np.argsort(arr["exch_ts"], kind="mergesort"), np.argsort(arr["local_ts"], kind="mergesort")
    )

    validate_event_order(arr)

    return arr

hft_array = get_hftbacktest_array(df_ob)

In [ ]:
hft_array

In [ ]:
from hftbacktest import BacktestAsset, HashMapMarketDepthBacktest, HashMapMarketDepthBacktest_TypeHint
from numba import njit

import numpy as np

@njit
def print_3depth(hbt: HashMapMarketDepthBacktest_TypeHint) -> bool:
    while hbt.elapse(60 * 1e9) == 0:
        print('current_timestamp:', hbt.current_timestamp)

        # Gets the market depth for the first asset, in the same order as when you created the backtest.
        depth = hbt.depth(0)

        # a key of bid_depth or ask_depth is price in ticks.
        # (integer) price_tick = price / tick_size
        i = 0
        for tick_price in range(depth.best_ask_tick, depth.best_ask_tick + 100):
            qty = depth.ask_qty_at_tick(tick_price)
            if qty > 0:
                print(
                    'ask: ',
                    qty,
                    '@',
                    np.round(tick_price * depth.tick_size, 1)
                )

                i += 1
                if i == 3:
                    break
        i = 0
        for tick_price in range(depth.best_bid_tick, max(depth.best_bid_tick - 100, 0), -1):
            qty = depth.bid_qty_at_tick(tick_price)
            if qty > 0:
                print(
                    'bid: ',
                    qty,
                    '@',
                    np.round(tick_price * depth.tick_size, 1)
                )

                i += 1
                if i == 3:
                    break
    return True

def get_ob_sampled(
        hbt: HashMapMarketDepthBacktest_TypeHint,
        delta: int,
        levels: int
) -> np.ndarray:
    
    # This function will be JIT-compiled by Numba
    @njit
    def _get_ob_sampled_njit(hbt, delta, levels):
        # Pre-allocate a buffer for the data
        capacity = 100_000  # Initial capacity
        
        timestamps = np.empty(capacity, dtype=np.int64)
        ask_prices = np.full((capacity, levels), np.nan, dtype=np.float64)
        ask_qtys = np.full((capacity, levels), np.nan, dtype=np.float64)
        bid_prices = np.full((capacity, levels), np.nan, dtype=np.float64)
        bid_qtys = np.full((capacity, levels), np.nan, dtype=np.float64)
        
        row_num = 0

        while hbt.elapse(delta) == 0:
            if row_num >= capacity:
                # Resize the array if capacity is exceeded
                new_capacity = capacity * 2
                
                new_timestamps = np.empty(new_capacity, dtype=np.int64)
                new_timestamps[:capacity] = timestamps
                timestamps = new_timestamps

                new_ask_prices = np.full((new_capacity, levels), np.nan, dtype=np.float64)
                new_ask_prices[:capacity] = ask_prices
                ask_prices = new_ask_prices

                new_ask_qtys = np.full((new_capacity, levels), np.nan, dtype=np.float64)
                new_ask_qtys[:capacity] = ask_qtys
                ask_qtys = new_ask_qtys

                new_bid_prices = np.full((new_capacity, levels), np.nan, dtype=np.float64)
                new_bid_prices[:capacity] = bid_prices
                bid_prices = new_bid_prices

                new_bid_qtys = np.full((new_capacity, levels), np.nan, dtype=np.float64)
                new_bid_qtys[:capacity] = bid_qtys
                bid_qtys = new_bid_qtys

                capacity = new_capacity

            depth = hbt.depth(0)
            tick_size = depth.tick_size
            
            timestamps[row_num] = hbt.current_timestamp

            # Collect ask levels
            lvl = 0
            if depth.best_ask_tick > 0:
                for tick_price in range(depth.best_ask_tick, depth.best_ask_tick + 10_000):
                    qty = depth.ask_qty_at_tick(tick_price)
                    if qty > 0:
                        ask_prices[row_num, lvl] = tick_price * tick_size
                        ask_qtys[row_num, lvl] = qty
                        lvl += 1
                        if lvl == levels:
                            break
            
            # Collect bid levels
            lvl = 0
            if depth.best_bid_tick > 0:
                for tick_price in range(depth.best_bid_tick, max(depth.best_bid_tick - 10_000, 0), -1):
                    qty = depth.bid_qty_at_tick(tick_price)
                    if qty > 0:
                        bid_prices[row_num, lvl] = tick_price * tick_size
                        bid_qtys[row_num, lvl] = qty
                        lvl += 1
                        if lvl == levels:
                            break
            
            row_num += 1
            
        return (
            timestamps[:row_num],
            ask_prices[:row_num],
            ask_qtys[:row_num],
            bid_prices[:row_num],
            bid_qtys[:row_num]
        )

    timestamps, ask_prices, ask_qtys, bid_prices, bid_qtys = _get_ob_sampled_njit(hbt, delta, levels)

    # Define the dtype for the structured array
    dtype = [('timestamp', 'i8')]
    for i in range(1, levels + 1):
        dtype.extend([(f'ask{i}_price', 'f8'), (f'ask{i}_qty', 'f8')])
    for i in range(1, levels + 1):
        dtype.extend([(f'bid{i}_price', 'f8'), (f'bid{i}_qty', 'f8')])
    
    # Merge the arrays into a single structured array
    num_rows = len(timestamps)
    structured_array = np.empty(num_rows, dtype=dtype)
    
    structured_array['timestamp'] = timestamps
    for i in range(levels):
        structured_array[f'ask{i+1}_price'] = ask_prices[:, i]
        structured_array[f'ask{i+1}_qty'] = ask_qtys[:, i]
        structured_array[f'bid{i+1}_price'] = bid_prices[:, i]
        structured_array[f'bid{i+1}_qty'] = bid_qtys[:, i]
        
    return structured_array

asset = (
    BacktestAsset()
        .data([hft_array])
        .linear_asset(1.0)
        .risk_adverse_queue_model()
        .no_partial_fill_exchange()
        .trading_value_fee_model(0.0002, 0.0007)
        .tick_size(0.1)
        .lot_size(0.001)
)

hbt = HashMapMarketDepthBacktest([asset])

sampled_ob = get_ob_sampled(hbt, delta=100 * 1e6, levels=5)

_ = hbt.close()

In [ ]:
df_sampled_ob = pl.DataFrame(sampled_ob)
df_sampled_ob